In [1]:
# 4.2 PENGUMPULAN DAN PEMAHAMAN DATA

# Mengimpor library
import pandas as pd
from pathlib import Path

# Menentukan lokasi dataset
csv_path = Path("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Alternatif jika working directory berada di folder utama
if not csv_path.exists():
    csv_path = Path("data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Membaca dataset
df = pd.read_csv(csv_path)

# Menampilkan ukuran dataset
print("Jumlah baris dan kolom:", df.shape)

# Menampilkan 5 baris pertama
display(df.head())

Jumlah baris dan kolom: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# Memeriksa tipe data setiap kolom
df.info()

# Memeriksa jumlah missing values
display(df.isnull().sum().to_frame("Missing Values"))

# Memeriksa duplikasi
print("Duplikasi baris:", df.duplicated().sum())
print("Duplikasi customerID:", df["customerID"].duplicated().sum())

# Memeriksa distribusi target
display(df["Churn"].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

,Missing Values
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Duplikasi baris: 0
Duplikasi customerID: 0


Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [3]:
# 4.2 - PEMERIKSAAN NILAI KOSONG TERSEMBUNYI

# Memeriksa string kosong atau hanya spasi
blank_counts = {
    col: df[col].astype("string").str.strip().eq("").sum()
    for col in df.select_dtypes(include="object").columns
}

blank_report = pd.DataFrame(
    blank_counts.items(),
    columns=["Column", "Blank Values"]
)

display(blank_report)

# Menguji konversi TotalCharges menjadi numerik
total_numeric = pd.to_numeric(
    df["TotalCharges"].astype("string").str.strip(),
    errors="coerce"
)

# Menampilkan baris yang gagal dikonversi
invalid_total = df.loc[
    total_numeric.isna(),
    ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]
]

print("Jumlah TotalCharges tidak valid:", len(invalid_total))
display(invalid_total)

# Memeriksa tenure pelanggan tersebut
print("Distribusi tenure:")
display(invalid_total["tenure"].value_counts())

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12896\1123713241.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns


,Column,Blank Values
0,customerID,0
1,gender,0
2,Partner,0
3,Dependents,0
4,PhoneService,0
5,MultipleLines,0
6,InternetService,0
7,OnlineSecurity,0
8,OnlineBackup,0
9,DeviceProtection,0


Jumlah TotalCharges tidak valid: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


Distribusi tenure:


tenure
0    11
Name: count, dtype: int64

In [4]:
# 4.3 - DATA PREPROCESSING

import numpy as np

# Salin data asli agar tidak berubah
clean = df.copy()

# Membersihkan spasi pada variabel teks
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].str.strip()

# Konversi TotalCharges
clean["TotalCharges"] = pd.to_numeric(
    clean["TotalCharges"],
    errors="coerce"
)

# Identifikasi nilai kosong
missing_total = clean["TotalCharges"].isna()

# Isi nol hanya jika tenure = 0
valid_zero = missing_total & (clean["tenure"] == 0)

clean.loc[valid_zero, "TotalCharges"] = 0.0

print("TotalCharges diperbaiki:", valid_zero.sum())

# Hentikan proses jika masih ada nilai tidak valid
if clean["TotalCharges"].isna().any():
    raise ValueError(
        "Masih ada TotalCharges kosong yang perlu diperiksa."
    )

# Pastikan customerID unik
if clean["customerID"].duplicated().any():
    raise ValueError("Terdapat customerID duplikat.")

# Mengubah status churn ke representasi biner
clean["ChurnBinary"] = clean["Churn"].map({
    "No": 0,
    "Yes": 1
})

if clean["ChurnBinary"].isna().any():
    raise ValueError("Terdapat kategori Churn tidak dikenal.")

# Feature engineering: TenureGroup
clean["TenureGroup"] = pd.cut(
    clean["tenure"],
    bins=[-1, 12, 24, 48, np.inf],
    labels=["0-12", "13-24", "25-48", "49+"]
)

# Menghitung layanan tambahan aktif
additional_services = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

clean["NumAdditionalServices"] = (
    clean[additional_services] == "Yes"
).sum(axis=1)

# Menghitung seluruh komponen layanan aktif
clean["NumTotalServices"] = (
    clean["NumAdditionalServices"]
    + (clean["PhoneService"] == "Yes").astype(int)
    + (clean["MultipleLines"] == "Yes").astype(int)
    + (clean["InternetService"] != "No").astype(int)
)

# Mengidentifikasi kontrak jangka panjang
clean["HasLongTermContract"] = clean["Contract"].isin(
    ["One year", "Two year"]
).astype(int)

# Kelompok biaya untuk EDA deskriptif saja
clean["MonthlyChargeGroup"] = pd.qcut(
    clean["MonthlyCharges"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print("Ukuran data setelah preprocessing:", clean.shape)
print("Total missing values:", clean.isna().sum().sum())

display(clean.head())

TotalCharges diperbaiki: 11
Ukuran data setelah preprocessing: (7043, 27)
Total missing values: 0


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12896\3053951723.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in clean.select_dtypes(include="object").columns:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,ChurnBinary,TenureGroup,NumAdditionalServices,NumTotalServices,HasLongTermContract,MonthlyChargeGroup
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Electronic check,29.85,29.85,No,0,0-12,1,2,0,Q1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Mailed check,56.95,1889.50,No,0,25-48,2,4,1,Q2
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Mailed check,53.85,108.15,Yes,1,0-12,2,4,0,Q2
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Bank transfer (automatic),42.30,1840.75,No,0,25-48,3,4,1,Q2
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Electronic check,70.70,151.65,Yes,1,0-12,0,2,0,Q3


In [5]:
# EKSPOR DATA PREPROCESSING

from pathlib import Path

# Folder hasil preprocessing
output_dir = csv_path.parent.parent / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Dataset utama untuk EDA, statistik, dan klasifikasi
clean.to_csv(
    output_dir / "telco_cleaned.csv",
    index=False
)

# Dataset khusus clustering (belum distandardisasi)
cluster_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "NumAdditionalServices",
    "NumTotalServices"
]

clean[
    ["customerID"] + cluster_features
].to_csv(
    output_dir / "telco_clustering_features.csv",
    index=False
)

# Ringkasan kualitas data
quality_report = pd.DataFrame({
    "Indicator": [
        "Jumlah pelanggan",
        "Jumlah variabel",
        "Duplikasi customerID",
        "Missing TotalCharges",
        "Missing keseluruhan"
    ],
    "Before": [
        len(df),
        df.shape[1],
        df["customerID"].duplicated().sum(),
        total_numeric.isna().sum(),
        df.isna().sum().sum()
    ],
    "After": [
        len(clean),
        clean.shape[1],
        clean["customerID"].duplicated().sum(),
        clean["TotalCharges"].isna().sum(),
        clean.isna().sum().sum()
    ]
})

quality_report.to_csv(
    output_dir / "preprocessing_report.csv",
    index=False
)

print("Semua file berhasil disimpan di:")
print(output_dir.resolve())
display(quality_report)

Semua file berhasil disimpan di:
C:\Users\LENOVO\OneDrive\customer churn\data\processed


,Indicator,Before,After
0,Jumlah pelanggan,7043,7043
1,Jumlah variabel,21,27
2,Duplikasi customerID,0,0
3,Missing TotalCharges,11,0
4,Missing keseluruhan,0,0


In [6]:
# CELL 6 - FINAL DATA QUALITY REPORT

# Menghitung nilai kosong termasuk string berisi spasi
blank_before = sum(
    df[col].astype("string").str.strip().eq("").sum()
    for col in df.select_dtypes(include="object").columns
)

missing_before = df.isna().sum().sum() + blank_before
missing_after = clean.isna().sum().sum()

quality_report = pd.DataFrame({
    "Indicator": [
        "Jumlah pelanggan",
        "Jumlah variabel",
        "Duplikasi customerID",
        "TotalCharges kosong/tidak valid",
        "Missing keseluruhan termasuk blank"
    ],
    "Before": [
        len(df),
        df.shape[1],
        df["customerID"].duplicated().sum(),
        total_numeric.isna().sum(),
        missing_before
    ],
    "After": [
        len(clean),
        clean.shape[1],
        clean["customerID"].duplicated().sum(),
        clean["TotalCharges"].isna().sum(),
        missing_after
    ]
})

quality_report.to_csv(
    output_dir / "preprocessing_report.csv",
    index=False
)

display(quality_report)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_12896\3561221101.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns


,Indicator,Before,After
0,Jumlah pelanggan,7043,7043
1,Jumlah variabel,21,27
2,Duplikasi customerID,0,0
3,TotalCharges kosong/tidak valid,11,0
4,Missing keseluruhan termasuk blank,11,0


In [7]:
# CELL 7 - DATA DICTIONARY

descriptions = {
    "customerID": "Identitas unik pelanggan",
    "gender": "Jenis kelamin pelanggan",
    "SeniorCitizen": "Indikator pelanggan senior (0/1)",
    "Partner": "Status memiliki pasangan",
    "Dependents": "Status memiliki tanggungan",
    "tenure": "Lama berlangganan dalam bulan",
    "PhoneService": "Status layanan telepon",
    "MultipleLines": "Status layanan beberapa saluran telepon",
    "InternetService": "Jenis layanan internet",
    "OnlineSecurity": "Status layanan keamanan online",
    "OnlineBackup": "Status layanan pencadangan online",
    "DeviceProtection": "Status perlindungan perangkat",
    "TechSupport": "Status dukungan teknis",
    "StreamingTV": "Status layanan streaming TV",
    "StreamingMovies": "Status layanan streaming film",
    "Contract": "Jenis kontrak pelanggan",
    "PaperlessBilling": "Status tagihan tanpa kertas",
    "PaymentMethod": "Metode pembayaran",
    "MonthlyCharges": "Biaya bulanan pelanggan",
    "TotalCharges": "Akumulasi tagihan pelanggan",
    "Churn": "Status churn: Yes atau No",
    "ChurnBinary": "Status churn numerik: Yes=1, No=0",
    "TenureGroup": "Kelompok lama berlangganan",
    "NumAdditionalServices": "Jumlah layanan tambahan aktif",
    "NumTotalServices": "Jumlah komponen layanan aktif",
    "HasLongTermContract": "Indikator kontrak jangka panjang",
    "MonthlyChargeGroup": "Kelompok kuartil biaya bulanan"
}

data_dictionary = pd.DataFrame({
    "Variable": clean.columns,
    "DataType": [str(clean[c].dtype) for c in clean.columns],
    "Description": [
        descriptions.get(c, "Perlu dokumentasi")
        for c in clean.columns
    ],
    "MissingValues": [
        int(clean[c].isna().sum())
        for c in clean.columns
    ]
})

data_dictionary.to_csv(
    output_dir / "data_dictionary.csv",
    index=False
)

display(data_dictionary)

print("Jumlah variabel:", len(data_dictionary))

,Variable,DataType,Description,MissingValues
0,customerID,str,Identitas unik pelanggan,0
1,gender,str,Jenis kelamin pelanggan,0
2,SeniorCitizen,int64,Indikator pelanggan senior (0/1),0
3,Partner,str,Status memiliki pasangan,0
4,Dependents,str,Status memiliki tanggungan,0
5,tenure,int64,Lama berlangganan dalam bulan,0
6,PhoneService,str,Status layanan telepon,0
7,MultipleLines,str,Status layanan beberapa saluran telepon,0
8,InternetService,str,Jenis layanan internet,0
9,OnlineSecurity,str,Status layanan keamanan online,0


Jumlah variabel: 27


In [8]:
# CELL 8 - FINAL VALIDATION

# Baca kembali file yang sudah diekspor
final_df = pd.read_csv(
    output_dir / "telco_cleaned.csv"
)

cluster_df = pd.read_csv(
    output_dir / "telco_clustering_features.csv"
)

# Pemeriksaan otomatis
assert len(final_df) == len(df)
assert final_df["customerID"].is_unique
assert final_df.isna().sum().sum() == 0
assert final_df["ChurnBinary"].isin([0, 1]).all()
assert len(cluster_df) == len(final_df)
assert "Churn" not in cluster_df.columns
assert "ChurnBinary" not in cluster_df.columns

# Periksa jumlah data target
print("Distribusi Churn:")
display(final_df["Churn"].value_counts())

print("Ukuran dataset utama:", final_df.shape)
print("Ukuran dataset clustering:", cluster_df.shape)

print("\nVALIDASI BERHASIL")
print("Dataset siap dibagikan kepada anggota kelompok.")

Distribusi Churn:


Churn
No     5174
Yes    1869
Name: count, dtype: int64

Ukuran dataset utama: (7043, 27)
Ukuran dataset clustering: (7043, 6)

VALIDASI BERHASIL
Dataset siap dibagikan kepada anggota kelompok.
